**Importing Neccessary Libraries**

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler

import xgboost as xgb

print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# List all files in the data directory to verify the presence of the dataset
path = "/content/drive/MyDrive/predictive-maintenance-smart-factory/data/"

list_of_files = os.listdir(path)
print("Files in the data directory:", list_of_files)

Files in the data directory: ['ai4i2020.csv', 'predictive_maintenance.csv', 'merged_predictive_maintenance.csv', 'processed_predictive_maintenance.csv']


In [9]:
# Load the dataset we created in feature engineering



df = pd.read_csv(os.path.join("/content/drive/MyDrive/predictive-maintenance-smart-factory/data/", 'processed_predictive_maintenance.csv'))

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Quick check of target distribution
print("\nMachine failure distribution:")
print(df['Machine failure'].value_counts(normalize=True))

Dataset shape: (10000, 55)

Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Failure Type', 'cumulative_tool_wear', 'time_since_last_tool_change', 'running_tool_wear', 'Torque [Nm]_rolling_mean_5', 'Torque [Nm]_rolling_std_5', 'Torque [Nm]_rolling_max_5', 'Torque [Nm]_rolling_mean_10', 'Torque [Nm]_rolling_std_10', 'Torque [Nm]_rolling_max_10', 'Rotational speed [rpm]_rolling_mean_5', 'Rotational speed [rpm]_rolling_std_5', 'Rotational speed [rpm]_rolling_max_5', 'Rotational speed [rpm]_rolling_mean_10', 'Rotational speed [rpm]_rolling_std_10', 'Rotational speed [rpm]_rolling_max_10', 'Process temperature [K]_rolling_mean_5', 'Process temperature [K]_rolling_std_5', 'Process temperature [K]_rolling_max_5', 'Process temperature [K]_rolling_mean_10', 'Process temperature [K]_rolling_std_10', 'Process temperature [K]_rolling_max_10', 'A

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 55 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   UDI                                      10000 non-null  int64  
 1   Product ID                               10000 non-null  object 
 2   Type                                     10000 non-null  object 
 3   Air temperature [K]                      10000 non-null  float64
 4   Process temperature [K]                  10000 non-null  float64
 5   Rotational speed [rpm]                   10000 non-null  int64  
 6   Torque [Nm]                              10000 non-null  float64
 7   Tool wear [min]                          10000 non-null  int64  
 8   Machine failure                          10000 non-null  int64  
 9   TWF                                      10000 non-null  int64  
 10  HDF                                      10000 

**Defining Features and Target**

In [11]:
# Define the target
target = 'Machine failure'

# Define features: exclude identifiers, target, and failure type columns
exclude_cols = ['Machine failure', 'Failure Type','Product ID', 'UDI', 
                'Type', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

feature_cols = [col for col in df.columns if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print("\nFirst 10 features:")
print(feature_cols[:10])

X = df[feature_cols]
y = df[target]

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

Number of features: 45

First 10 features:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'cumulative_tool_wear', 'time_since_last_tool_change', 'running_tool_wear', 'Torque [Nm]_rolling_mean_5', 'Torque [Nm]_rolling_std_5']

Feature matrix shape: (10000, 45)
Target shape: (10000,)


In [12]:
from sklearn.model_selection import TimeSeriesSplit

# We will use 5 splits for time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

print("TimeSeriesSplit created with 5 splits")

# Let's also create a simple hold-out set (last 20% of data as test set)
train_size = int(len(df) * 0.8)
X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Failure rate in train: {y_train.mean():.4f}")
print(f"Failure rate in test: {y_test.mean():.4f}")

TimeSeriesSplit created with 5 splits

Training set shape: (8000, 45)
Test set shape: (2000, 45)
Failure rate in train: 0.0375
Failure rate in test: 0.0195


Why TimeSeriesSplit instead of train_test_split?
Because this is time-series / IIoT data.
In a real factory, you cannot use future data to predict the past.
TimeSeriesSplit respects the time order — it trains on earlier data and validates on later data.

**Training Random Forest Baseline**

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'   # Helps with imbalanced data
)

# Train on the training set
rf_model.fit(X_train, y_train)

print( "Random Forest model trained successfully!")

Random Forest model trained successfully!


**Evaluating the Model**

In [14]:
from sklearn.metrics import classification_report, roc_auc_score

# Make predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Print results
print("=== Random Forest Performance ===")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

=== Random Forest Performance ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1961
           1       0.80      0.62      0.70        39

    accuracy                           0.99      2000
   macro avg       0.90      0.81      0.85      2000
weighted avg       0.99      0.99      0.99      2000


ROC-AUC Score: 0.9788

Confusion Matrix:
[[1955    6]
 [  15   24]]


The random forest model here is strong but not so strong, the recall rate is unacceptable. 

Now, trying to improve on it

In [15]:
# Improved Random Forest
rf_improved = RandomForestClassifier(
    n_estimators=500,           # more trees = more stable
    max_depth=None,             # let trees grow deeper
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',    # already good
    random_state=42,
    n_jobs=-1                   # use all CPU cores
)

rf_improved.fit(X_train, y_train)

# Evaluate again
y_pred = rf_improved.predict(X_test)
y_pred_proba = rf_improved.predict_proba(X_test)[:, 1]

print("=== Improved Random Forest ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

=== Improved Random Forest ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1961
           1       0.76      0.64      0.69        39

    accuracy                           0.99      2000
   macro avg       0.88      0.82      0.84      2000
weighted avg       0.99      0.99      0.99      2000

ROC-AUC: 0.9787

Confusion Matrix:
[[1953    8]
 [  14   25]]


1. The number number of decision trees was increased, the decision tree is allowed to grow deeper, and all cpu cores was used to train the model, in order to bring good performace.
The recall moves from 62% to 64% but  reduced the precision from 80% to 76%

In [16]:
from sklearn.metrics import precision_recall_curve

# Get precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find threshold that gives better recall while keeping reasonable precision
optimal_idx = np.argmax(recalls * precisions)   # simple way
optimal_threshold = thresholds[optimal_idx]

print(f"Best threshold found: {optimal_threshold:.3f}")

# Apply new threshold
y_pred_custom = (y_pred_proba >= optimal_threshold).astype(int)

print("\n=== With Custom Threshold ===")
print(classification_report(y_test, y_pred_custom))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_custom)    
print("\nConfusion Matrix:")
print(cm)

Best threshold found: 0.519

=== With Custom Threshold ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1961
           1       0.81      0.64      0.71        39

    accuracy                           0.99      2000
   macro avg       0.90      0.82      0.85      2000
weighted avg       0.99      0.99      0.99      2000


Confusion Matrix:
[[1955    6]
 [  14   25]]


2. Another improvement made was to tune the threshold, the threshold was tuned to finding the optimal threshold that gives better recall with a reasonable precision.
The Recall here is 64% and the Precision is 81% which is both better. This is a small improvement but the Recall is not high enough for a real factory system.

**Training XGBOOST Model**

In [18]:
# Clean feature names - remove problematic characters
X_train.columns = [str(col).replace('[', '_').replace(']', '').replace('<', '').replace('>', '') 
                   for col in X_train.columns]

X_test.columns = [str(col).replace('[', '_').replace(']', '').replace('<', '').replace('>', '') 
                  for col in X_test.columns]

print("Feature names cleaned successfully!")
print("First 10 cleaned feature names:")
print(X_train.columns[:10].tolist())

Feature names cleaned successfully!
First 10 cleaned feature names:
['Air temperature _K', 'Process temperature _K', 'Rotational speed _rpm', 'Torque _Nm', 'Tool wear _min', 'cumulative_tool_wear', 'time_since_last_tool_change', 'running_tool_wear', 'Torque _Nm_rolling_mean_5', 'Torque _Nm_rolling_std_5']


In [19]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ====================== XGBoost Model ======================

# Initialize XGBoost classifier with good default parameters for imbalanced data
xgb_model = xgb.XGBClassifier(
    n_estimators=500,           # Number of trees
    max_depth=6,                # Depth of each tree
    learning_rate=0.05,         # Step size - smaller is better but slower
    subsample=0.8,              # Use 80% of data for each tree
    colsample_bytree=0.8,       # Use 80% of features for each tree
    random_state=42,
    eval_metric='auc',          # Optimize for AUC during training
    scale_pos_weight=25,
    n_jobs=-1                   # Use all available CPU cores
)

# Train the model
xgb_model.fit(
    X_train, 
    y_train,
    eval_set=[(X_test, y_test)],   # Monitor performance on test set
    verbose=True                 # Set to True to see training progress
)

print("XGBoost model trained successfully!")

[0]	validation_0-auc:0.89928
[1]	validation_0-auc:0.95224
[2]	validation_0-auc:0.95635
[3]	validation_0-auc:0.94085
[4]	validation_0-auc:0.93885
[5]	validation_0-auc:0.93185
[6]	validation_0-auc:0.93469
[7]	validation_0-auc:0.93926
[8]	validation_0-auc:0.92969
[9]	validation_0-auc:0.94536
[10]	validation_0-auc:0.94425
[11]	validation_0-auc:0.94378
[12]	validation_0-auc:0.94333
[13]	validation_0-auc:0.94294
[14]	validation_0-auc:0.94281
[15]	validation_0-auc:0.95220
[16]	validation_0-auc:0.95195
[17]	validation_0-auc:0.96136
[18]	validation_0-auc:0.96083
[19]	validation_0-auc:0.96526
[20]	validation_0-auc:0.96528
[21]	validation_0-auc:0.96453
[22]	validation_0-auc:0.96408
[23]	validation_0-auc:0.96415
[24]	validation_0-auc:0.96500
[25]	validation_0-auc:0.96598
[26]	validation_0-auc:0.96477
[27]	validation_0-auc:0.96455
[28]	validation_0-auc:0.96523
[29]	validation_0-auc:0.96472
[30]	validation_0-auc:0.96381
[31]	validation_0-auc:0.96301
[32]	validation_0-auc:0.96411
[33]	validation_0-au

**Making Evaluation of XGBOOST**

In [20]:
# Make predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost Performance ===")
print(classification_report(y_test, y_pred_xgb))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("\nConfusion Matrix:")
print(cm_xgb)

=== XGBoost Performance ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1961
           1       0.70      0.72      0.71        39

    accuracy                           0.99      2000
   macro avg       0.85      0.86      0.85      2000
weighted avg       0.99      0.99      0.99      2000


ROC-AUC Score: 0.9549

Confusion Matrix:
[[1949   12]
 [  11   28]]


## Model Comparison: Random Forest vs XGBoost

After training both models on the same engineered features using a chronological train-test split, here is the performance comparison:

### Key Results (Failure Class - Class 1)

| Model              | Precision | Recall | F1-Score | ROC-AUC | False Negatives | False Positives |
|--------------------|-----------|--------|----------|---------|-----------------|-----------------|
| Random Forest      | 0.81      | 0.64   | 0.71     | 0.9788  | 15              | ~6              |
| **XGBoost**        | **0.70**  | **0.72**| **0.71** | 0.9549  | **11**          | 12              |

### Observations:

- **XGBoost** achieved a **higher Recall (0.72)** compared to the best Random Forest (0.64).  
  This means XGBoost is better at **catching actual machine failures**, which is the most critical metric in predictive maintenance.

- Random Forest showed slightly better Precision and ROC-AUC, meaning it was more conservative and produced fewer false alarms.

- Both models performed well on the majority class (No Failure), but struggled relatively with the minority class (Failure), which is expected due to severe class imbalance (~3.5% failures).

- **Conclusion**: XGBoost is currently the stronger model for this task because it reduces the number of missed failures (False Negatives)


## Why Hyperparameter Tuning with Optuna?

While both Random Forest and XGBoost delivered decent baseline performance, there is still significant room for improvement, especially in **Recall** for the failure class.

Default or manually chosen hyperparameters often do not give the best possible performance. Hyperparameter tuning helps us systematically find the optimal combination of parameters such as:
- `n_estimators`, `max_depth`, `learning_rate`
- `subsample`, `colsample_bytree`, `scale_pos_weight`

### What Optuna Can Do:

- **Automated & Intelligent Search**: Optuna uses advanced techniques (Bayesian Optimization) to intelligently explore the hyperparameter space instead of trying every possible combination (which would be extremely slow).
- **Improve Model Performance**: It often leads to noticeable gains in Recall and F1-score.
- **Reduce Overfitting**: Finds the right balance between model complexity and generalization.
- **Save Time**: Much more efficient than manual tuning or Grid Search.

In this project, we will use Optuna to tune the XGBoost model with the goal of further **increasing Recall** while maintaining acceptable Precision.

**Performing OPTUNA Tunning**